# 💧 SSCAP Data Analysis Dashboard

This notebook provides a complete pipeline to download, process, and analyze data from your SSCAP API deployment. It handles geographic localization via IP, timezone discovery, and advanced date parsing for local time strings.

In [ ]:
import pandas as pd
import requests
import json
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import MarkerCluster
from datetime import datetime
import time
import pytz
import re

# Set style for better plots
plt.style.use('fivethirtyeight')
sns.set_palette("viridis")

def is_iso_format(date_str):
    """
    Checks if a string is in a proper ISO format with timezone info.
    Examples: 2024-03-20T10:00:00Z, 2024-03-20T10:00:00-06:00
    """
    if not isinstance(date_str, str):
        return False
    # Check for 'T' separator and either 'Z' or offset (+/-)
    return 'T' in date_str and (date_str.endswith('Z') or re.search(r'[+-]\d{2}:?\d{2}$', date_str))

def parse_smart_date(date_val, tz_name='UTC'):
    """
    Parses a date value smartly.
    If it's already ISO, parse as UTC/Absolute.
    If it's a 'local text' string (e.g., 2024-03-20 10:00:00), assume it's in tz_name.
    """
    if pd.isna(date_val) or date_val is None:
        return None, False
    
    date_str = str(date_val).strip()
    is_converted = False
    
    try:
        if is_iso_format(date_str):
            # Proper ISO with TZ info
            dt = pd.to_datetime(date_str, utc=True)
        else:
            # Assume local text if no TZ info found
            is_converted = True
            dt_naive = pd.to_datetime(date_str)
            local_tz = pytz.timezone(tz_name)
            dt = local_tz.localize(dt_naive).astimezone(pytz.UTC)
            
        return dt, is_converted
    except Exception as e:
        # Fallback to pandas default parsing
        return pd.to_datetime(date_val, errors='coerce'), False

def download_sscap_data(api_url, ip_tz_map=None):
    """
    Downloads data and processes it with timezone awareness.
    """
    api_url = api_url.rstrip('/')
    download_url = f"{api_url}/api/download" if not api_url.endswith('/api/download') else api_url
    
    print(f"Fetching data from {download_url}...")
    response = requests.get(download_url)
    response.raise_for_status()
    
    records = response.json().get('data', [])
    print(f"Processing {len(records)} records...")
    
    flattened = []
    for record in records:
        filename = record.get('filename', '')
        record_type = filename.split('/')[1] if '/' in filename else 'unknown'
        content = record.get('content', {})
        data = content.get('data', {})
        metadata = content.get('metadata', {})
        ip = metadata.get('ip')
        
        # Get timezone for this IP if available
        tz = ip_tz_map.get(ip, {}).get('timezone', 'UTC') if ip_tz_map else 'UTC'
        
        # Smart Date Parsing for catched_at / used_at
        raw_date = data.get('catched_at') or data.get('used_at')
        parsed_dt, converted = parse_smart_date(raw_date, tz)
        
        flat = {
            'id': content.get('id'),
            'type': record_type,
            'uploaded_at': pd.to_datetime(record.get('uploadedAt'), utc=True),
            'timestamp': parsed_dt,
            'raw_timestamp': raw_date,
            'is_local_text_conversion': converted,
            'timezone_assumed': tz,
            **data,
            'meta_ip': ip,
            'meta_city': metadata.get('city', 'Unknown'),
            'meta_country': metadata.get('country', 'Unknown')
        }
        flattened.append(flat)
        
    return pd.DataFrame(flattened)

## 🌍 1. IP Geolocation & Timezone Discovery
We first fetch the server data to get the IPs, then discovery their location and timezones.

In [ ]:
API_URL = "https://sscap-api.vercel.app" # @param {type:"string"}

def discover_ips(api_url):
    resp = requests.get(f"{api_url.rstrip('/')}/api/download")
    records = resp.json().get('data', [])
    ips = list(set([r.get('content', {}).get('metadata', {}).get('ip') for r in records if r.get('content', {}).get('metadata', {}).get('ip')]))
    return ips

def geocode_and_tz(ips):
    results = {}
    for ip in ips:
        if ip == '127.0.0.1': continue
        try:
            print(f"Discovering location/TZ for {ip}...")
            r = requests.get(f"http://ip-api.com/json/{ip}?fields=status,message,country,city,lat,lon,timezone", timeout=5)
            d = r.json()
            if d.get('status') == 'success':
                results[ip] = d
                print(f"  -> {d.get('city')}, {d.get('timezone')}")
            time.sleep(1.0)
        except Exception as e:
            print(f"  Error: {e}")
    return results

unique_ips = discover_ips(API_URL)
ip_map = geocode_and_tz(unique_ips)

# Now download and process full dataframe
df = download_sscap_data(API_URL, ip_map)
df.head()

## 🗺️ 2. Geographic Map
Visualize the devices. Markers include hover tooltips and detailed popups.

In [ ]:
if not df.empty:
    # Add lat/lon from our discovery map
    df['lat'] = df['meta_ip'].map(lambda x: ip_map.get(x, {}).get('lat'))
    df['lon'] = df['meta_ip'].map(lambda x: ip_map.get(x, {}).get('lon'))
    
    geo_df = df.dropna(subset=['lat', 'lon']).copy()
    
    if not geo_df.empty:
        m = folium.Map(tiles='CartoDB Positron')
        cluster = MarkerCluster().add_to(m)
        bounds = []
        
        for _, row in geo_df.iterrows():
            # Determine value to display (meters vs pulses)
            val = row.get('meters', row.get('pulses', 'N/A'))
            unit = 'M' if row['type'] == 'nivel' else 'Pulses'
            
            # Debug print for pulses
            if row['type'] in ['captado', 'utilizado']:
                print(f"Debug: {row['type']} record from {row['meta_city']} shows {val} pulses")
            
            popup = f"""
            <div style='font-family: sans-serif; font-size: 12px; min-width: 160px;'>
                <b style='color:#2a9d8f;'>{row['type'].upper()}</b> ({row['tlaloque_id']})<br>
                <b>Location:</b> {row['meta_city']}, {row['meta_country']}<br>
                <b>Time:</b> {row['timestamp'].strftime('%Y-%m-%d %H:%M %Z') if pd.notnull(row['timestamp']) else 'N/A'}<br>
                <hr style='margin:5px 0;'>
                <b style='font-size:14px;'>{val} {unit}</b><br>
                <small style='color:grey;'>Converted from local text: {row['is_local_text_conversion']}</small>
            </div>
            """
            folium.Marker(
                [row['lat'], row['lon']],
                popup=folium.Popup(popup, max_width=300),
                tooltip=f"{row['type'].capitalize()}: {val} {unit}"
            ).add_to(cluster)
            bounds.append([row['lat'], row['lon']])
            
        m.fit_bounds(bounds, padding=(50,50))
        display(m)
    else:
        print("No geocodable IPs found.")

## 📊 3. Time Series Analysis
Analysis of levels and captured water, now normalized to UTC with local conversion tracking.

In [ ]:
# Filter for NIVEL
df_nivel = df[df['type'] == 'nivel'].dropna(subset=['timestamp', 'meters']).sort_values('timestamp')

if not df_nivel.empty:
    plt.figure(figsize=(15, 6))
    plt.plot(df_nivel['timestamp'], df_nivel['meters'], color='#2a9d8f', label='Measured Level')
    plt.scatter(df_nivel[df_nivel['is_local_text_conversion']]['timestamp'], 
                df_nivel[df_nivel['is_local_text_conversion']]['meters'], 
                color='#e76f51', s=30, label='Local Time Converted', zorder=5)
    
    plt.title('Water Level (UTC Time)', fontsize=16)
    plt.ylabel('Meters', fontsize=12)
    plt.legend()
    plt.show()
else:
    print("No water level data.")